# Dr.Plant — تجربة تطوير NGLD V5

**بحثية فقط، لا تستبدل النموذج الأصلي.** تستخدم التجربة صور التدريب دون تسميات (تعلم ذاتي بمنظرين + تقطير معرفة من المعلم المجمد)، ثم تقارن المعلم والمرشح على اختبار NGLD المحجوز مرة واحدة. النتائج لا تثبت الأداء الحقلي السوري.

## 1) ربط Google Drive
ارفع إلى جذر My Drive الملفين: `DrPlant-NGLD-Research-Bundle-2026-11-25.zip` ودفتر Colab هذا. ثم شغّل الخلايا بالترتيب. يلزم GPU مجاني متاح؛ إذا لم يتوفر GPU أوقف التجربة ولا تشغّلها على CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import glob, os, zipfile, shutil, subprocess, sys, json
BUNDLE_NAME = 'DrPlant-NGLD-Research-Bundle-2026-11-25.zip'
matches = list(Path('/content/drive/MyDrive').rglob(BUNDLE_NAME))
assert matches, f'لم أجد {BUNDLE_NAME} في My Drive'
BUNDLE = matches[0]
WORK = Path('/content/drplant_ngld_research')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as z:
    bad = z.testzip(); assert bad is None, bad
    z.extractall(WORK)
ROOT = WORK / 'ngld-colab-bundle'
print('Bundle:', BUNDLE, 'size:', BUNDLE.stat().st_size)
print('Extracted:', ROOT)

In [ ]:
import torch, torchvision
print('torch', torch.__version__, 'torchvision', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('لم تتوفر GPU في هذه الجلسة. لا تتابع على CPU؛ أعد المحاولة لاحقاً بعد اختيار GPU إذا كان متاحاً مجاناً.')
print(torch.cuda.get_device_name(0))


In [ ]:
# تثبيت الحزم الصغيرة الناقصة فقط؛ لا نعيد تنزيل PyTorch في بيئة Colab
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'pillow'], check=True)


In [ ]:
# استخراج الصور وفحص المسارات قبل التدريب
source_zip = ROOT / 'source' / 'NGLD-V5.zip'
images_dir = WORK / 'dataset'
images_dir.mkdir()
with zipfile.ZipFile(source_zip) as z:
    bad = z.testzip(); assert bad is None, bad
    z.extractall(images_dir)
roots = list(images_dir.rglob('Grapes Disease Dataset'))
assert len(roots) == 1, f'Expected one image root, found: {roots}'
IMAGE_ROOT = roots[0]
MANIFEST = ROOT / 'training-manifest.csv'
TEACHER = ROOT / 'model' / 'teacher-best-model.pt'
print('IMAGE_ROOT:', IMAGE_ROOT)
print('manifest:', MANIFEST.stat().st_size, 'teacher:', TEACHER.stat().st_size)

In [ ]:
# تحقق شامل قبل استهلاك GPU: المسارات، SHA-256، الأعداد، وعزل المجموعات
import csv, hashlib
rows = list(csv.DictReader(MANIFEST.open(encoding='utf-8-sig')))
assert len(rows) == 2389, f'Unexpected manifest row count: {len(rows)}'
missing = []
bad_hashes = []
for i, r in enumerate(rows, 1):
    p = IMAGE_ROOT / r['relative_path']
    if not p.is_file():
        missing.append(r['relative_path']); continue
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    if h.hexdigest().lower() != r['sha256'].strip().lower():
        bad_hashes.append(r['relative_path'])
    if i % 500 == 0: print('SHA checked:', i)
assert not missing, f'Missing images: {len(missing)}'
assert not bad_hashes, f'SHA-256 mismatches: {len(bad_hashes)}'
expected = {'train': 1433, 'validation': 478, 'test': 478}
splits = {s: sum(r['split'] == s for r in rows) for s in expected}
assert splits == expected, f'Unexpected split sizes: {splits}'
groups = {s: {r['group_id_for_split'] for r in rows if r['split'] == s} for s in expected}
assert not (groups['train'] & groups['validation'] or groups['train'] & groups['test'] or groups['validation'] & groups['test']), 'Group leakage detected'
classes = sorted({r['class_label'] for r in rows})
assert classes == ['Bacterial Leaf Spot', 'Downy Mildew', 'Healthy Leaves', 'Powdery Mildew'], classes
print('PASS: images=', len(rows), 'splits=', splits, 'classes=', classes)
print('Group counts:', {s: len(groups[s]) for s in groups})
print('expert_label_review:', sorted({r['expert_label_review'] for r in rows}))
print('Note: labels are not expert-reviewed; the training script ignores labels for its self-supervised loss.')


In [ ]:
# تدريب مرشح مستقل: لا يكتب فوق المعلم الأصلي
SCRIPT = ROOT / 'scripts' / 'train_ngld_self_supervised_distillation.py'
OUT = Path('/content/drplant_ngld_candidate')
cmd = [sys.executable, str(SCRIPT), str(IMAGE_ROOT), str(MANIFEST), str(TEACHER), '--out', str(OUT), '--epochs', '8', '--batch-size', '32', '--learning-rate', '2e-5', '--unfreeze-last-blocks', '2', '--contrastive-weight', '0.15', '--distillation-weight', '1.0', '--temperature', '2.0', '--image-size', '160', '--seed', '42']
print('Starting:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# عرض المقارنة والتحقق من سلامة المرشح وحفظ حزمة النتائج على Drive
metrics = json.loads((OUT / 'metrics.json').read_text())
assert metrics['status'] == 'research_candidate_not_for_app'
assert metrics['teacher_checkpoint_sha256'] == __import__('hashlib').sha256(TEACHER.read_bytes()).hexdigest()
print(json.dumps({k: metrics[k] for k in ['method','teacher_checkpoint_sha256','manifest_sha256','teacher_test','candidate_test','delta_accuracy','delta_macro_f1','limitations'] if k in metrics}, ensure_ascii=False, indent=2))
RESULTS = Path('/content/DrPlant-NGLD-Candidate-Results-2026-11-25.zip')
with zipfile.ZipFile(RESULTS, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(OUT))
DEST = Path('/content/drive/MyDrive') / RESULTS.name
shutil.copy2(RESULTS, DEST)
print('Saved:', DEST, 'bytes:', DEST.stat().st_size)


## ضوابط قراءة النتيجة
- هذا تحديث تمثيلات بحثي على صور التدريب غير الموسومة، مع إبقاء المعلم مجمداً لتقليل النسيان.
- تسميات NGLD ليست مراجعة من خبير؛ لذلك لا نفسر نتيجة الاختبار على أنها تشخيص حقلي موثوق.
- لا ندمج المرشح بالتطبيق حتى لو تحسن الرقم؛ يلزم اختبار مستقل من مزارع/أشجار/جلسات تصوير مختلفة، ومعايرة ورفض الحالات المجهولة.
- إذا تراجع المرشح، نحتفظ بالمعلم ولا نرقّيه.